# Sila na koljeno: predvidi → izračunaj → provjeri

Promatramo stacionaran tok kroz horizontalno koljeno. Tlakovi su **manometarski**, a presječne normale usmjerene su iz kontrolnog volumena. Račun najprije daje silu stijenke na fluid; sila fluida na koljeno ima suprotan smjer.

## Predvidi

1. Koja komponenta količine gibanja mijenja predznak kada kut prijeđe \(90^\circ\)?
2. Za ravnu cijev jednakih promjera i tlakova, smije li ostati rezultantna sila?
3. Nacrtaj vektore brzine, tlačne sile i nepoznate reakcije prije računanja komponenti.

Osnovni račun koristi podatke Z3. Kut β u notebooku pozitivan je od +x prema +y u tlocrtu; P6 zato se poziva s β = −60°. Tlakovi su neovisni ulazi i ne nameće se tok bez gubitaka.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

def elbow_balance(beta_deg, Q, D1, D2, p1_gauge, p2_gauge, rho=998.0):
    beta = np.deg2rad(beta_deg)
    e1 = np.array([1.0, 0.0])
    e2 = np.array([np.cos(beta), np.sin(beta)])
    A1, A2 = np.pi*D1**2/4, np.pi*D2**2/4
    V1, V2 = (Q/A1)*e1, (Q/A2)*e2
    mdot = rho*Q

    momentum_change = mdot*(V2-V1)
    pressure_on_fluid = p1_gauge*A1*e1 - p2_gauge*A2*e2
    wall_on_fluid = momentum_change - pressure_on_fluid
    fluid_on_elbow = -wall_on_fluid
    closure = pressure_on_fluid + wall_on_fluid - momentum_change
    return {
        "A1": A1, "A2": A2, "V1": V1, "V2": V2, "mdot": mdot,
        "momentum_change": momentum_change,
        "pressure_on_fluid": pressure_on_fluid,
        "wall_on_fluid": wall_on_fluid,
        "fluid_on_elbow": fluid_on_elbow,
        "closure": closure,
    }

case = elbow_balance(
    beta_deg=90.0, Q=0.026, D1=0.100, D2=0.100,
    p1_gauge=180_000.0, p2_gauge=150_000.0,
)
for key in ["momentum_change", "pressure_on_fluid", "wall_on_fluid", "fluid_on_elbow", "closure"]:
    print(f"{key:20s} = [{case[key][0]:9.2f}, {case[key][1]:9.2f}] N")
print(f"|F_fluid→koljeno| = {np.linalg.norm(case['fluid_on_elbow']):.2f} N")


## Izračunaj: vektorska bilanca i osjetljivost na kut

Za kontrolni volumen vrijedi

\[
\mathbf F_p+\mathbf F_{stijenka\to fluid}
=\dot m(\mathbf V_2-\mathbf V_1).
\]

Rezidual zatvaranja je razlika lijeve i desne strane. Mora biti mali u obje komponente, ne samo po iznosu rezultante. Zatim mijenjamo kut uz nepromijenjene ostale ulaze.


In [ ]:
angles = np.linspace(0, 180, 181)
forces = np.array([
    elbow_balance(b, 0.026, 0.100, 0.100, 180_000.0, 150_000.0)["fluid_on_elbow"]
    for b in angles
])
magnitudes = np.linalg.norm(forces, axis=1)

straight = elbow_balance(0.0, 0.025, 0.100, 0.100, 180_000.0, 180_000.0)
u_turn_zero_pressure = elbow_balance(180.0, 0.025, 0.100, 0.100, 0.0, 0.0)
expected_u_turn_x = 2*u_turn_zero_pressure["mdot"]*np.linalg.norm(u_turn_zero_pressure["V1"])

print(f"Najveći iznos u promatranom rasponu: {magnitudes.max():.1f} N pri β={angles[magnitudes.argmax()]:.0f}°")
print(f"Rezidual osnovnog slučaja: {np.linalg.norm(case['closure']):.3e} N")


## Provjeri

Provjeravamo zatvaranje vektorske bilance, nul-slučaj ravne jednolike cijevi i analitički rezultat za okret od \(180^\circ\) bez tlačnih sila.


In [ ]:
assert np.linalg.norm(case["closure"]) < 1e-9
assert np.linalg.norm(straight["fluid_on_elbow"]) < 1e-9
assert np.isclose(u_turn_zero_pressure["fluid_on_elbow"][0], expected_u_turn_x, rtol=1e-12)
assert abs(u_turn_zero_pressure["fluid_on_elbow"][1]) < 1e-9
assert np.isclose(case["A1"]*np.linalg.norm(case["V1"]), case["A2"]*np.linalg.norm(case["V2"]), rtol=1e-13)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(angles, forces[:,0], label="$F_x$")
axes[0].plot(angles, forces[:,1], label="$F_y$")
axes[0].plot(angles, magnitudes, "k--", label="$|F|$")
axes[0].set(xlabel=r"kut koljena $\beta$ (°)", ylabel="sila fluida na koljeno (N)", title="Osjetljivost na smjer izlaza")
axes[0].legend()

vectors = [case["pressure_on_fluid"], case["wall_on_fluid"], -case["momentum_change"]]
names = ["tlak", "stijenka", "−promjena količine gibanja"]
colors = ["#256d85", "#b43c35", "#2d7d46"]
origin = np.zeros(2)
scale = max(np.linalg.norm(v) for v in vectors)
for vector, name, color in zip(vectors, names, colors):
    axes[1].quiver(*origin, *vector, angles="xy", scale_units="xy", scale=1, color=color, label=name)
    origin = origin + vector
axes[1].set_xlim(-1.2*scale, 1.2*scale); axes[1].set_ylim(-1.2*scale, 1.2*scale)
axes[1].set_aspect("equal"); axes[1].set(xlabel="$x$ (N)", ylabel="$y$ (N)", title="Poligon zatvaranja sila")
axes[1].legend(fontsize=8)
for ax in axes: ax.grid(True, ls=":", alpha=.45)
plt.tight_layout(); plt.show()


## Provjera predznaka P6 te zadatci Z4 i Z5

Predvidi smjer sile kada se tok skreće prema $-y$. Zatim za ekscentrični mlaz
razlikuj okomiti krak od udaljenosti po x. Za jednu ploču koja se udaljava
predvidi masu vode koja do nje stiže u sekundi i snagu pri $u=0$ i $u=v$.
Koji kontrolni volumen ostaje stacionaran u sustavu ploče?


In [ ]:
# Z3 i P6: golden rezultati provjeravaju i predznake komponenata.
assert np.allclose(case["fluid_on_elbow"], [1499.61554535, -1263.99609633], atol=1e-7, rtol=0)
p6 = elbow_balance(-60.0, .18, .20, .20, 280e3, 280e3)
assert p6["fluid_on_elbow"][0] > 0 and p6["fluid_on_elbow"][1] > 0
assert np.isclose(np.degrees(np.arctan2(*p6["fluid_on_elbow"][::-1])), 60.0, atol=1e-10)

# Z4: sila iz količine gibanja, zatim moment na krutoj konstrukciji.
rho, Q, v, b, e = 998.0, .016, 18.0, .20, .35
Fx, Fy = rho*Q*v, 0.0
moment_z = b*Fy-e*Fx
assert np.isclose(moment_z, -100.5984, atol=1e-10)
assert np.isclose((b+.50)*Fy-e*Fx, moment_z, atol=1e-10)
print(f"Z4: Fx={Fx:.4f} N; moment mlaza={moment_z:.4f} N m")

# Z5: dotok kroz pomičnu granicu; impuls i energija koriste apsolutne brzine.
d, v, u = .040, 20.0, 8.0
A = np.pi*d*d/4
def moving_plate(u):
    mdot = rho*A*(v-u)
    force = mdot*(v-u)
    outlet_speed = np.hypot(u, v-u)
    power = force*u
    energy_power = .5*mdot*(v*v-outlet_speed*outlet_speed)
    return mdot, force, power, outlet_speed, energy_power

mdot, force, power, v2, energy_power = moving_plate(u)
assert np.allclose([mdot, force, power, v2],
                   [15.04948545, 180.59382537, 1444.75060298, 14.42220510], atol=1e-7, rtol=0)
assert np.isclose(power, energy_power, atol=1e-9, rtol=0)
assert np.isclose(rho*A*v, mdot+rho*A*u, atol=1e-12, rtol=0)
speeds = np.linspace(0, v, 301)
values = moving_plate(speeds)
assert np.allclose(values[2], values[4], atol=1e-9, rtol=0)
assert values[2][0] == values[2][-1] == 0
assert np.isclose(speeds[np.argmax(values[2])], v/3)
print(f"Z5: mdot_rel={mdot:.6f} kg/s; F={force:.6f} N; P={power:.6f} W")
print(f"Z5: u_opt={v/3:.6f} m/s; P_max={moving_plate(v/3)[2]:.6f} W")
fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.plot(speeds, values[2]/1000, label="jedna ploča: F u")
ax.axvline(v/3, color="#c0392b", ls="--", label="u = v/3")
ax.set(xlabel="brzina ploče u (m/s)", ylabel="snaga predana ploči (kW)")
ax.grid(True, ls=":", alpha=.45); ax.legend(); plt.tight_layout(); plt.show()


## Protumači

Rezultantna sila nije jednaka samo promjeni količine gibanja: tlačne sile mogu biti dominantne. Za vertikalno koljeno bilanci bi trebalo dodati težinu fluida u kontrolnom volumenu i jasno navesti je li težina samog koljena dio promatranog sustava.


1. Zašto zamjena b ne mijenja moment u Z4, a promjena e ga mijenja?
2. Zašto jaka sila na nepomičnu ploču ne znači i veliku predanu snagu?
3. Gdje završava razlika između protoka iz sapnice i protoka koji stiže do
   ploče koja se udaljava? Kako to potvrđuje masena bilanca?
